# How Evaluation Datasets Are Created

In procurement RAG systems, you can't evaluate what you can't measure. An evaluation dataset is your **ground truth** — a curated set of question/answer pairs that tells you whether your system is actually working.

---

## Why Production Teams Create Test Questions

The goal isn't to test the LLM in general. It's to test whether **your specific system**, on **your specific documents**, handles **your actual users' real needs**.

Production teams build eval datasets because:

- **Coverage** — you need to know the system works across all document types (RFPs, bid submissions, contracts, compliance checklists)
- **Regression testing** — when you update your chunking strategy or swap your retrieval model, you need to know if things got better or worse
- **Edge case tracking** — real users will ask weird, multi-part, or adversarial questions; you need those in your eval set
- **Stakeholder confidence** — procurement committees need evidence the system is reliable before trusting it with ₹10 Cr decisions

> Think of it like a QA checklist before a tender goes live. You wouldn't publish an RFP without someone checking it. You shouldn't deploy a RAG system without someone checking it either.

---

## Good Test Questions vs. Bad Ones

This is where most teams go wrong. A large eval dataset full of bad questions gives you false confidence.

### What Good Questions Look Like

| Property | What It Means | Example |
|---|---|---|
| **Specific** | Answerable from one or a few known passages | *"What is the EMD amount required in RFP-2024-IT-047?"* |
| **Grounded** | The answer exists in a document you can point to | Answer: *"₹2,00,000 — Section 4.2"* |
| **Representative** | Reflects what real users actually ask | Drawn from procurement officer queries or helpdesk logs |
| **Appropriately difficult** | Tests real failure modes, not just obvious facts | *"Which vendors submitted bids but failed mandatory eligibility?"* |
| **Unambiguous** | Has one defensible correct answer | Not open to interpretation |

### What Bad Questions Look Like

| Anti-pattern | Example | Why It's a Problem |
|---|---|---|
| **Too vague** | *"Tell me about Vendor A"* | No single correct answer; can't evaluate |
| **Answerable without retrieval** | *"What does EMD stand for?"* | Tests general knowledge, not your documents |
| **Multiple correct answers** | *"What are the payment terms?"* (across 6 vendor contracts) | Evaluator can't grade it fairly |
| **Unanswerable from docs** | *"Why did Vendor B lose the last tender?"* | Answer isn't in any document |
| **Leading** | *"Doesn't Vendor C have the best price?"* | Presupposes the answer |

---

## How Synthetic Generation Works

Manually writing hundreds of question/answer pairs is slow and expensive. Synthetic generation lets you **prompt an LLM against your source documents** to produce Q&A pairs at scale.

### The Basic Pipeline

```
Source Documents
      │
      ▼
  Chunking
  (split into passages of ~300–500 tokens)
      │
      ▼
  LLM Prompted per Chunk
  ("Given this passage, generate 3 questions a procurement officer might ask,
    along with the correct answer drawn only from the passage")
      │
      ▼
  Raw Q&A Pairs
  (large volume, variable quality)
      │
      ▼
  Quality Filtering
      │
      ▼
  Final Eval Dataset
```

### The Prompt Pattern

Here's what a synthetic generation prompt looks like in practice:

```
You are building an evaluation dataset for a procurement RAG system.

Given the document passage below, generate 3 question-answer pairs that:
- A procurement officer or bid evaluator might realistically ask
- Are answerable strictly from the passage — do not use outside knowledge
- Have a single, unambiguous correct answer
- Vary in type: one factual lookup, one comparative or conditional, one
  requiring inference across two sentences

Return JSON: [{"question": "...", "answer": "...", "source_span": "..."}]

PASSAGE:
[Section 6.1 of RFP-2024-IT-047]
Vendors must submit an Earnest Money Deposit of ₹2,00,000 in the form of a
demand draft drawn in favour of the Chief Finance Officer. Bids submitted
without a valid EMD will be summarily rejected without further evaluation.
EMD will be refunded to unsuccessful bidders within 30 days of contract
award.
```

### What It Generates

```json
[
  {
    "question": "What is the required EMD amount for RFP-2024-IT-047?",
    "answer": "₹2,00,000",
    "source_span": "Vendors must submit an Earnest Money Deposit of ₹2,00,000"
  },
  {
    "question": "What happens to a bid submitted without a valid EMD?",
    "answer": "It is summarily rejected without further evaluation.",
    "source_span": "Bids submitted without a valid EMD will be summarily rejected"
  },
  {
    "question": "Within how many days will EMD be refunded to unsuccessful bidders?",
    "answer": "Within 30 days of contract award.",
    "source_span": "EMD will be refunded to unsuccessful bidders within 30 days of contract award"
  }
]
```

You run this against every chunk across your entire document corpus — potentially generating thousands of pairs from hundreds of tender documents.

---

## Quality Filtering

Raw synthetic output is noisy. You must filter before these questions enter your eval dataset.

### What to Remove

**1. Ambiguous questions** — where the answer depends on which document or vendor you're referring to

> *"What is the delivery timeline?"* — without specifying the vendor or contract, this is unanswerable

Filter: Flag questions that don't contain a specific entity (vendor name, RFP ID, section number) when the corpus has multiple matching documents.

**2. Duplicate or near-duplicate questions** — the LLM will often generate slight variations of the same question from adjacent chunks

> *"When is EMD refunded?"* and *"How soon is EMD returned to unsuccessful vendors?"* — same question, different phrasing

Filter: Run semantic similarity scoring (e.g., cosine similarity > 0.92 → deduplicate, keep one).

**3. Unanswerable questions** — where the LLM hallucinated a question the passage doesn't actually support

> Passage mentions vendor registration. LLM generates: *"What penalty applies if vendor registration lapses?"* — the passage says nothing about penalties.

Filter: Run a verification pass — prompt a second LLM: *"Can this question be answered from this passage alone? Yes/No."* Drop all No's.

**4. Trivially easy questions** — so obvious they add no signal

> *"Does Section 6.1 exist in this document?"*

Filter: Heuristic — questions under 6 words, or questions whose answer is a single word like "Yes" or "No", are candidates for removal unless they're genuinely important compliance checks.

**5. Format-broken outputs** — malformed JSON, missing answer field, source span that doesn't match the passage

Filter: Schema validation at parse time; any pair that fails validation is dropped.

### Filtering Summary

| Filter | Method | When to Apply |
|---|---|---|
| Ambiguity | Entity check + human review | Before adding to dataset |
| Duplicates | Semantic similarity threshold | After full generation run |
| Unanswerable | LLM verification pass | Before human review |
| Trivial | Length + answer heuristics | After generation |
| Malformed | Schema validation | At parse time, immediately |

A well-filtered synthetic dataset typically **retains 55–75% of raw generated pairs**. If you're retaining 95%, your filters are too loose.

---

## Where Manual Review Is Still Necessary

Synthetic generation scales. Manual review makes it trustworthy. You cannot fully replace one with the other.

### 1. Seeding Edge Cases the LLM Won't Think Of

LLMs generate plausible, common questions. They won't generate adversarial or unusual ones.

Humans must add:
- *"Vendor D's bid is dated after the submission deadline — are they eligible?"* (deadline edge case)
- *"Two vendors quoted the same price — how does the tiebreaker clause apply?"* (conflict scenario)
- *"The RFP specifies ISO 9001:2015, but Vendor E has ISO 9001:2008 — does this qualify?"* (version ambiguity)

These are exactly the questions where RAG systems fail. They won't appear in synthetic sets without deliberate human effort.

### 2. Verifying Gold Answers

The LLM writes its own answers. It can be confidently wrong.

> Synthetic answer: *"EMD will be refunded within 30 days."*
> Actual clause: *"within 30 working days"* — a legally meaningful difference in procurement.

A human reviewer must read the source passage and confirm the answer is not just approximately correct but **precisely correct**, especially for numbers, dates, deadlines, and eligibility conditions.

### 3. Calibrating Difficulty Distribution

Synthetic generation naturally overproduces easy factual lookups because most passages contain simple extractable facts. A human curator must ensure the dataset has a healthy spread:

| Difficulty | Type | Target % |
|---|---|---|
| Easy | Direct fact extraction | ~30% |
| Medium | Multi-sentence synthesis | ~40% |
| Hard | Cross-document reasoning / edge cases | ~30% |

Without curation, you end up with 80% easy questions — and a system that scores 90% on your eval but fails constantly in production.

### 4. Domain Expert Sign-Off

Procurement has legal and regulatory nuance. A general LLM doesn't know that *"technically compliant"* and *"substantially compliant"* mean different things in public tender law, or that GeM portal rules differ from standard CPPP procedures.

A procurement SME (subject matter expert) reviewing 10–15% of the dataset catches category errors that no automated filter will find.

---

## End-to-End Summary

```
Step 1 — Synthetic Generation
  LLM + source chunks → thousands of raw Q&A pairs (fast, cheap, scalable)

Step 2 — Automated Filtering
  Remove ambiguous, duplicate, unanswerable, trivial, malformed → keep ~65%

Step 3 — Human Curation
  Verify gold answers, add edge cases, fix difficulty distribution

Step 4 — Domain Expert Review
  Spot-check 10–15% for legal/regulatory correctness

Step 5 — Locked Eval Dataset
  Versioned, stable — used for regression testing across all system changes
```

The ratio in mature teams is roughly **70% synthetic, 30% human-added or human-verified**. Neither alone is sufficient — synthetic generation without human review produces a dataset that teaches you your system is better than it is.